In [ ]:
import sys
import cv2
import torch
from PyQt5.QtWidgets import *
from PyQt5.QtGui import *
from PyQt5.QtCore import *

# ================== 你的模型路径 ==================
YOLO_MODEL_PATH = r'D:\project\yolov5\runs\train\exp2\weights\best.pt'
YOLO_REPO_PATH = r'D:\project\yolov5'

# ================== 主窗口 ==================
class MainWindow(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("无人机检测系统")
        self.setGeometry(200, 200, 1000, 600)

        self.img_path = None
        self.init_ui()

    def init_ui(self):
        # 按钮
        self.btn_open = QPushButton("打开图片")
        self.btn_detect = QPushButton("开始检测")

        # 模式选择
        self.combo = QComboBox()
        self.combo.addItems(["YOLO检测", "千问+YOLO检测"])

        # 图像显示
        self.label_src = QLabel("原始图像")
        self.label_res = QLabel("检测结果")

        self.label_src.setFixedSize(450, 400)
        self.label_res.setFixedSize(450, 400)

        # 布局
        h_top = QHBoxLayout()
        h_top.addWidget(self.btn_open)
        h_top.addWidget(self.combo)
        h_top.addWidget(self.btn_detect)

        h_img = QHBoxLayout()
        h_img.addWidget(self.label_src)
        h_img.addWidget(self.label_res)

        v_main = QVBoxLayout()
        v_main.addLayout(h_top)
        v_main.addLayout(h_img)

        self.setLayout(v_main)

        # 绑定事件
        self.btn_open.clicked.connect(self.open_image)
        self.btn_detect.clicked.connect(self.detect)

    # ================== 打开图片 ==================
    def open_image(self):
        file_path, _ = QFileDialog.getOpenFileName(self, "选择图片", "", "*.jpg *.png")
        if file_path:
            self.img_path = file_path
            img = cv2.imread(file_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            self.show_image(img, self.label_src)

    # ================== 显示图像 ==================
    def show_image(self, img, label):
        h, w, ch = img.shape
        qt_img = QImage(img.data, w, h, ch * w, QImage.Format_RGB888)
        label.setPixmap(QPixmap.fromImage(qt_img).scaled(label.size(), Qt.KeepAspectRatio))

    # ================== 检测逻辑 ==================
    def detect(self):
        if not self.img_path:
            QMessageBox.warning(self, "错误", "请先选择图片")
            return

        mode = self.combo.currentText()
        img = cv2.imread(self.img_path)

        if mode == "YOLO检测":
            result_img = self.yolo_detect(img)

        elif mode == "千问+YOLO检测":
            result_img = self.qwen_yolo_detect(img)

        result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
        self.show_image(result_img, self.label_res)

    # ================== YOLO检测 ==================
    def yolo_detect(self, img):
        model = torch.hub.load(YOLO_REPO_PATH, 'custom',
                               path=YOLO_MODEL_PATH, source='local')

        results = model(img)
        return results.render()[0]

    # ================== 千问 + YOLO ==================
    def qwen_yolo_detect(self, img):
        # ====== 这里替换成你的函数 ======
        # 示例：假装检测框
        h, w, _ = img.shape
        cv2.rectangle(img, (50, 50), (200, 200), (0, 255, 0), 2)
        return img


# ================== 运行 ==================
if __name__ == "__main__":
    app = QApplication(sys.argv)
    win = MainWindow()
    win.show()
    sys.exit(app.exec_())